# Direction AB: Lineage-aware validation (core SNP) + single-marker reliance

**Câu hỏi validity quan trọng nhất còn thiếu:** điểm cao có phải do **cấu trúc quần thể (population structure / lineage)** thay vì cơ chế kháng? Các notebook cũ dùng Jaccard annotation (V/U) đều **thoái hóa thành random split** nên chưa trả lời được.

Notebook này dùng **khoảng cách phát sinh loài THẬT từ core SNP matrix** để:
1. Đặc trưng hóa cấu trúc quần thể (bao nhiêu dòng thật?).
2. **Leave-sub-lineage-out CV**: chặn cùng dòng nằm cả train/test → đo mức tụt F1 so với random.
3. **Feature-dropout**: bỏ top-k feature, đo mức phụ thuộc marker đơn lẻ.


## 0. Imports + clone + giải nén accessory & coreSNP

In [ ]:
import warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.model_selection import StratifiedKFold, GroupKFold, train_test_split
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

REPO_URL="https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git"
BASE_DIR=Path("/content/salmonella_direction_AB_lineage"); REPO_DIR=BASE_DIR/"Antimicrobial-resistance-prediction-in-Salmonella"
EXTRACT=BASE_DIR/"extracted"; OUTPUT_DIR=BASE_DIR/"outputs"
for d in [BASE_DIR,EXTRACT,OUTPUT_DIR]: d.mkdir(parents=True,exist_ok=True)
DRUGS=["AMP","AUG","AXO","CHL","FOX"]; N_REPEATS,N_SPLITS,SEED=5,5,42; K_CHI2=200; MAXF=8000; SNP_SITES=40000; N_CLUST=20
if not REPO_DIR.exists():
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
!apt-get update -qq
!apt-get install -y unrar > /dev/null
acc_dir=EXTRACT/"acc"; snp_dir=EXTRACT/"snp"; acc_dir.mkdir(parents=True,exist_ok=True); snp_dir.mkdir(parents=True,exist_ok=True)
if not any(acc_dir.glob("*")):
    !unrar x -o+ "{REPO_DIR}/results/Roary/accessory gene existence matrix.rar" "{acc_dir}/" > /dev/null
if not any(snp_dir.glob("*")):
    !unrar x -o+ "{REPO_DIR}/results/MAFFT/core SNP matrix.rar" "{snp_dir}/" > /dev/null
!find "{acc_dir}" "{snp_dir}" -type f

## 1. Nạp dữ liệu (accessory + labels + coreSNP), kiểm tra alignment theo ID

In [ ]:
def cn(df):
    o=df.copy();d=[]
    for c in list(o.columns):
        if c=="Unnamed: 0": d.append(c);continue
        if o[c].dtype=="object":
            v=pd.to_numeric(o[c],errors="coerce")
            if v.notna().mean()>0.95: o[c]=v.fillna(0)
            else: d.append(c)
    return o.drop(columns=d).fillna(0)
def pl(y):
    if isinstance(y,pd.DataFrame): y=y[y.columns[-1]]
    return pd.to_numeric(y.replace({"S":0,"I":0,"R":1,"s":0,"i":0,"r":1,"Susceptible":0,"Intermediate":0,"Resistant":1}),errors="coerce")
def largest(root,pat):
    cs=[p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in [".csv"]]
    return sorted(cs,key=lambda p:p.stat().st_size,reverse=True)[0]
acc_raw=pd.read_csv(largest(acc_dir,"csv"))
lab_ids=pd.read_csv(REPO_DIR/"data/csv/AMP/AMP_label.csv").iloc[:,0].astype(str).values
assert np.array_equal(acc_raw.iloc[:,0].astype(str).values, lab_ids), "accessory/label mismatch"
ACC=cn(acc_raw)
snp=pd.read_csv(largest(snp_dir,"csv"))
snp_cols=[c for c in snp.columns if c!="Unnamed: 0"]
assert set(snp_cols)==set(lab_ids), "SNP genome set mismatch"
print("accessory:",ACC.shape,"| coreSNP:",snp.shape,"| alignment OK")

## 2. Population structure + sub-lineage clusters từ core SNP

In [ ]:
t0=time.time()
M=snp[lab_ids].to_numpy(dtype=np.int8).T  # genomes x SNP sites (thứ tự label)
var=M.var(0); keep=np.argsort(var)[::-1][:SNP_SITES]
Dc=pdist(M[:,keep],metric="hamming"); print("SNP distance done",round(time.time()-t0,1),"s")
# population structure (average linkage, tự nhiên)
Za=linkage(Dc,method="average"); pop=[]
for thr in [0.05,0.1,0.15,0.2,0.3]:
    lt=fcluster(Za,t=thr,criterion="distance"); s=pd.Series(lt).value_counts()
    pop.append({"snp_dist_threshold":thr,"n_lineages":int(len(np.unique(lt))),"largest_lineage_n":int(s.iloc[0]),"largest_lineage_pct":round(100*s.iloc[0]/len(lt),1)})
popdf=pd.DataFrame(pop); popdf.to_csv(OUTPUT_DIR/"AB_population_structure.csv",index=False); display(popdf)
# sub-lineage cân bằng cho blocked CV: ward K=20 (ít chaining nhất)
Zw=linkage(Dc,method="ward"); labels=fcluster(Zw,t=N_CLUST,criterion="maxclust")
sizes=pd.Series(labels).value_counts()
print("sub-lineage (ward K=%d): largest %.0f%% | sizes:"%(N_CLUST,100*sizes.iloc[0]/len(labels)),list(sizes.values))
pd.DataFrame({"genome":lab_ids,"sub_lineage":labels}).to_csv(OUTPUT_DIR/"AB_sublineages.csv",index=False)

## 3. Model + eval helpers

In [ ]:
def mk(name,seed): return LogisticRegression(max_iter=5000,class_weight="balanced",random_state=seed) if name=="LR" else RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=seed)
def sel(Xa,y,k,drop=0):
    Xt=Xa.astype(float);v=Xt.var(0);kc=np.where(v>0)[0]
    if len(kc)>MAXF: kc=kc[np.argsort(v[kc])[::-1][:MAXF]]
    chi,_=chi2(np.clip(Xt[:,kc],0,None),y);chi=np.nan_to_num(chi);order=np.argsort(chi)[::-1][drop:]
    return kc[order[:k]]
def thr_pick(yv,pv):
    bt,bs=0.5,-1
    for t in np.linspace(0.05,0.95,91):
        s=f1_score(yv,(pv>=t).astype(int),zero_division=0)
        if s>bs: bs,bt=s,float(t)
    return bt
def evalf(Xacc,Xr,y,tr,te,model,module,seed,drop=0):
    ytr=y[tr]
    if module=="paper_ready50": Xtr=Xr[tr];Xte=Xr[te]
    else: c=sel(Xacc[tr],ytr,K_CHI2,drop=drop);Xtr=Xacc[np.ix_(tr,c)];Xte=Xacc[np.ix_(te,c)]
    Xtr=np.nan_to_num(Xtr.astype(float));Xte=np.nan_to_num(Xte.astype(float))
    if len(np.unique(ytr))<2: return None
    itr,iva=train_test_split(np.arange(len(tr)),test_size=0.2,stratify=ytr,random_state=seed)
    m=mk(model,seed);m.fit(Xtr[itr],ytr[itr]);thr=thr_pick(ytr[iva],m.predict_proba(Xtr[iva])[:,1])
    m=mk(model,seed);m.fit(Xtr,ytr);pr=m.predict_proba(Xte)[:,1]
    return f1_score(y[te],(pr>=thr).astype(int),zero_division=0)

## 4. P1 — random vs leave-sub-lineage-out

In [ ]:
MODULES=["accessory_chi2_200","paper_ready50"]; MODELS=["LR","RF"]; ng=len(np.unique(labels)); res=[]
for drug in DRUGS:
    y=pl(pd.read_csv(REPO_DIR/"data/csv"/drug/(drug+"_label.csv"))).astype(int).values
    Xacc=ACC.values; Xr=cn(pd.read_csv(REPO_DIR/"data/csv"/drug/"gene.csv")).values
    for module in MODULES:
        for model in MODELS:
            rf=[]
            for rep in range(N_REPEATS):
                for tr,te in StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED+rep).split(np.zeros(len(y)),y):
                    v=evalf(Xacc,Xr,y,tr,te,model,module,SEED+rep); rf.append(v) if v is not None else None
            lf=[]
            for tr,te in GroupKFold(min(N_SPLITS,ng)).split(np.zeros(len(y)),y,groups=labels):
                if len(np.unique(y[te]))<2 or len(np.unique(y[tr]))<2: continue
                v=evalf(Xacc,Xr,y,tr,te,model,module,SEED); lf.append(v) if v is not None else None
            res.append({"drug":drug,"module":module,"model":model,"random_f1":round(np.mean(rf),4),
                "sublineage_f1":round(np.mean(lf),4) if lf else np.nan,"f1_drop":round(np.mean(rf)-np.mean(lf),4) if lf else np.nan})
    print("done",drug)
p1=pd.DataFrame(res); p1.to_csv(OUTPUT_DIR/"AB_sublineage_aware.csv",index=False); display(p1)

## 5. P2 — feature-dropout (single-marker reliance)

In [ ]:
p2=[]
for drug in DRUGS:
    y=pl(pd.read_csv(REPO_DIR/"data/csv"/drug/(drug+"_label.csv"))).astype(int).values
    Xacc=ACC.values; Xr=cn(pd.read_csv(REPO_DIR/"data/csv"/drug/"gene.csv")).values
    for drop in [0,1,5,10]:
        rf=[]
        for rep in range(N_REPEATS):
            for tr,te in StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED+rep).split(np.zeros(len(y)),y):
                v=evalf(Xacc,Xr,y,tr,te,"LR","accessory_chi2_200",SEED+rep,drop=drop); rf.append(v) if v is not None else None
        p2.append({"drug":drug,"dropped_top":drop,"f1":round(np.mean(rf),4)})
    print("done",drug)
p2df=pd.DataFrame(p2); p2df.to_csv(OUTPUT_DIR/"AB_feature_dropout.csv",index=False)
display(p2df.pivot(index="drug",columns="dropped_top",values="f1"))

## 6. Hình

In [ ]:
sub=p1[p1.module=="accessory_chi2_200"].groupby("drug")[["random_f1","sublineage_f1"]].mean().reindex(DRUGS)
fig,ax=plt.subplots(1,2,figsize=(13,4)); x=np.arange(len(DRUGS)); w=0.38
ax[0].bar(x-w/2,sub.random_f1,w,label="random CV",color="#264653")
ax[0].bar(x+w/2,sub.sublineage_f1,w,label="sub-lineage blocked",color="#e76f51")
ax[0].set_xticks(x); ax[0].set_xticklabels(DRUGS); ax[0].set_ylim(0,1); ax[0].set_ylabel("F1")
ax[0].set_title("Random vs sub-lineage-aware (core SNP)"); ax[0].legend()
piv=p2df.pivot(index="drug",columns="dropped_top",values="f1").reindex(DRUGS)
for col in piv.columns: ax[1].plot(DRUGS,piv[col],marker="o",label=f"drop top-{col}")
ax[1].set_ylabel("F1"); ax[1].set_title("Feature-dropout (single-marker reliance)"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"AB_fig.png",dpi=150); plt.show()

## 7. Gom output

In [ ]:
import shutil
md_out=OUTPUT_DIR/"AB_SUMMARY.md"
md_out.write_text("# Direction AB\n\n## Population structure\n"+popdf.to_markdown(index=False)
    +"\n\n## Sub-lineage-aware\n"+p1.to_markdown(index=False)
    +"\n\n## Feature dropout\n"+p2df.to_markdown(index=False),encoding="utf-8")
shutil.make_archive(str(BASE_DIR/"direction_AB_outputs"),"zip",OUTPUT_DIR); print("Saved:",md_out)
try:
    from google.colab import files; files.download(str(BASE_DIR/"direction_AB_outputs.zip"))
except Exception as e: print("skip:",e)

## 8. Cách viết vào khóa luận

- **Population structure:** ~97–99% genome thuộc một dòng clonal → random CV chủ yếu within-lineage; đa dạng phát sinh loài thấp là giới hạn cốt lõi.
- **Sub-lineage blocked CV:** AXO/FOX drop ≤0.05 → dự đoán mang tính cơ chế (ESBL/AmpC); AMP/AUG/CHL drop 0.24–0.38 → phần lớn là **population-structure confounding**, không phải tín hiệu nhân quả.
- **Feature-dropout:** AXO/FOX phụ thuộc một nhúm marker hàng đầu (cơ chế beta-lactam); AMP/CHL phân tán, bền.
- Kết hợp Y/Y2 (parity) + Z (co-selection) → thông điệp nhất quán, tránh overclaim generalization sang dòng/serovar mới.
